In [3]:
# train.py
import os
import json
from dataclasses import dataclass
from typing import List, Dict, Any
from functools import partial

import numpy as np
from datasets import load_dataset, Dataset, concatenate_datasets
from torch.utils.data import Dataset 
from transformers import (
    AutoTokenizer,
    AutoModel,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    Trainer,
AutoModelForCausalLM
)
from transformers.trainer_utils import get_last_checkpoint
from accelerate import Accelerator
import torch
from sklearn.model_selection import train_test_split
import torch.nn.functional as F
import torch.nn as nn
import pandas as pd

import torch

from dataclasses import dataclass
from typing import Dict, Any, Callable, Optional
import evaluate
import shutil
import glob
from datetime import datetime
from peft import LoraConfig, get_peft_model

import os
import argparse
import numpy as np
import torch
from datetime import datetime

from transformers import AutoTokenizer, TrainingArguments, Trainer
import sys
from pathlib import Path
import wandb

In [23]:
odf = pd.read_csv("full_babe_with_cats.csv")
odf['text'] = odf['text'].astype(str).fillna("")
odf["followed_news_outlets_clean"] = (
    odf["followed_news_outlets"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.replace("[", "", regex=False)
    .str.replace("]", "", regex=False)
    .str.replace("'", "", regex=False)
    .str.replace('"', "", regex=False)
)
odf["num_foll_news_outlet"] = odf["followed_news_outlets_clean"].apply(
    lambda x: len([item for item in x.split(",") if item.strip() != ""])
)

odf["num_age"] = pd.cut(odf["age"],bins=[-np.inf, 18, 30, 45, 55, 65, np.inf],labels=False)

odf["num_poli_idealogy"] = pd.cut(odf["political_ideology"],
                                 bins=[-np.inf, -6, -1, 1, 5, np.inf],
                                 labels=False)

odf["num_foll_news_outlet"] = pd.cut(odf["num_foll_news_outlet"],
                                          bins=[-np.inf, 1, 2, 3, 4, 6, np.inf],
                                          labels=False)


odf["num_news_check_frequency"] = odf["news_check_frequency"].map({
    "Never": 0,
    "Very rarely": 1,
    "Several times per month": 2,
    "Several times per week": 3,
    "Every day": 4,
    "Several times per day": 5,
})

def bin_education(edu):
    edu = str(edu).strip()
    if edu in {"Graduate work"}:
        return 5
    if edu in {"BA", "Bachelor", "Bachelors", "Bachelor's degree", "Bachelor’s degree"}:
        return 4
    if edu in {"Associate degree"}:
        return 3
    if edu in {"Some college", "Vocational or technical school"}:
        return 2
    if edu == "High school graduate":
        return 1
    return 0

odf["num_education"] = odf["education"].apply(bin_education)
odf['num_gender'] = odf['gender'].map({
    'Male': 1, 'male': 1,
    'Female': 0, 'female': 0,
}).fillna(2).astype(int)
odf["label_binary"] = odf["label_bias"].str.startswith("Biased").astype(int)


/scratch/slurm-2626453/ipykernel_1005519/603078894.py:1: DtypeWarning: Columns (0: survey_completed) have mixed types. Specify dtype option on import or set low_memory=False.
  odf = pd.read_csv("full_babe_with_cats.csv")


In [25]:
# odf.columns
# odf['education'].value_counts()
# odf[odf['num_education'] == 0]['education'].value_counts()
# odf['num_gender'].value_counts()
# odf['num_foll_news_outlet'].value_counts()
# for col in ["num_age", "num_gender", "num_education", "num_poli_idealogy", "news_check_frequency", "num_foll_news_outlet"]:
#     print(odf[col].value_counts())

# odf[['dataset', 'num_education']].value_counts()
odf.shape



(49734, 32)

In [60]:
odf['num_poli_idealogy'].value_counts()

num_poli_idealogy
0    17756
2    10053
1     8517
4     7046
3     6362
Name: count, dtype: int64

In [5]:
#embeds

cat_cardinalities = {
    "words": 11267,
    "factual": 3,
    "group_id": 85,
    "type": 3,
    "topic": 14,
    "outlet": 8,
    "age": 54,
    "gender": 3,
    "education": 8,
    "native_english_speaker": 3,
    "political_ideology": 21,
    "followed_news_outlets": 551,
    "news_check_frequency": 6,
    "num_foll_news_outlet": 6,
    "num_poli_idealogy": 6,
    "num_age": 6,
    "num_gender": 3,
    "num_education": 6,
}


out_cats = ["num_age", "num_gender", "num_education", "num_poli_idealogy", "num_news_check_frequency", "num_foll_news_outlet"]
in_cats = [k for k, card in cat_cardinalities.items() if k not in ["num_age", "num_gender", "num_education", "num_poli_idealogy", "num_news_check_frequency", "num_foll_news_outlet"]]
cat_embs_info = {}
for k, card in cat_cardinalities.items():
    if k not in odf.columns:
        continue
    emb_dim = max(2, int(card ** 0.5) + 1) #sqrt(cardinality)) + 1
    emb_dim = min(50, emb_dim) #but not more than 50 - cough cough words
    cat_embs_info[k] = {
        "cardinality": card, 
        "emb_dim": emb_dim,
        "out_cat": k in out_cats,
        "in_cat": k in in_cats}




class MTLDataset(Dataset):
    def __init__(self, 
                 df, 
                 cat_info, 
                 tokenizer,
                 zero_cats = False,
                 max_length=128):
        
        df = df.reset_index(drop=True).copy()
        self.texts = df["text"].tolist()
     
        for col in list(cat_info.keys()):
            df[col] = df[col].astype('category')
            try:
                df[col] = df[col].cat.codes
            except:
                print(col)

        cat_ids = df[list(cat_info.keys())].values   #[N, num_cat_vars]
        cat_ids = cat_ids.astype(int)

        if zero_cats:
            cat_ids = np.zeros_like(cat_ids)

    
        self.cat_ids = cat_ids
        self.labels = df["label_binary"].to_numpy(dtype=np.float32)
        self.out_cat_names = [k for k, v in cat_info.items() if v.get("out_cat", False)]
        self.out_cat_ids = {name: df[name].values for name in self.out_cat_names}
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        cats = self.cat_ids[idx]

        enc = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        item = {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "x_cats": torch.tensor(cats, dtype=torch.long),
            "labels": torch.tensor(self.labels[idx], dtype=torch.float),
            "target_cats": torch.tensor(
                [self.out_cat_ids[name][idx] for name in self.out_cat_names],
                dtype=torch.long
            ),
        }
        return item
    
    


In [6]:
class ambiProbe(nn.Module):

    
    def __init__(self, 
                cats_info, 
                llm_pretrained_name
                ):
        super().__init__()

        self.llm = AutoModel.from_pretrained(
            llm_pretrained_name,
            output_hidden_states=True
        )
        for p in self.llm.parameters():
            p.requires_grad = False
        
        self.llm.eval()
                

        d_model = self.llm.config.hidden_size

        late_fusion_width = d_model + 1 #plus 1 for the label
        self.linear_post_fusion = nn.Linear(late_fusion_width, d_model)

        
        self.shared_trunk = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(),
            nn.Linear(d_model // 2, d_model // 4),
        )
        
        self.out_cat_names = [k for k, v in cats_info.items() if v.get("out_cat", False)]
        self.probe_heads = nn.ModuleDict({
            name: nn.Linear(d_model // 4, cats_info[name]["cardinality"])
            for name in self.out_cat_names
        })

    def forward(
        self,
        input_ids, #embedding of input text
        attention_mask,
        x_cats,
        labels, # [B] binary silver labels
        target_cats  # [B, out_cats] numerical features
    ):

        outputs = self.llm(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        #uncollate here instead of building a collator for the dataset?
        target_cats_dict = {}
        for idx, name in enumerate(self.out_cat_names):
            target_cats_dict[name] = target_cats[:, idx]

        last_hidden = outputs.last_hidden_state # [B, L, d_model]
        mask = attention_mask.unsqueeze(-1)            # [B, L, 1]
        hidden_masked = last_hidden * mask             # [B, L, d_model]
        sum_hidden = hidden_masked.sum(dim=1)          # [B, d_model]
        lengths = mask.sum(dim=1).clamp(min=1)         # [B, 1]
        h_text = sum_hidden / lengths                  # [B, d_model]
        

        h_fused = torch.cat([h_text, labels.unsqueeze(-1).float()], dim=-1) # [B, d_model + 1]
        h_proj = self.linear_post_fusion(h_fused)    # [B, d_model]
        h_shared = self.shared_trunk(h_proj)         # [B, shared_hdim]

        

        logits = {name: head(h_shared) for name, head in self.probe_heads.items()}

        for name in self.probe_heads:
            target = target_cats_dict[name]
            logit = logits[name]
            n_classes = logit.shape[-1]


            bad = (target < 0) | (target >= n_classes)
            if bad.any():
                bad_vals = target[bad].detach().cpu().tolist()[:20]
                raise ValueError(
                    f"Bad targets for head {name}: {bad_vals}; "
                    f"valid range is [0, {n_classes - 1}]"
                )


                
        losses = {name: F.cross_entropy(logits[name], target_cats_dict[name]) 
                  for name in self.probe_heads}

        total_loss = torch.stack(list(losses.values())).sum()
        

        return {
            "loss": total_loss,
            "piecewise_losses": losses,
            "out_logits" : logits
        }
        

In [7]:
class MTLClassifier(nn.Module):
    #mtl model, two heads.
    #shared base and trunk, then binary bias head. this is fed back into head 2
    # head 2 has num_target_cats heads, and gets cross entropy loss with the probe. it actually outputs an average tho.
    
    def __init__(self, 
                cats_info, #dict of all cats and cardianlities and stuff
                llm_pretrained_name, 
                use_lora, 
                lora_r,
                ambi_loss_weight,
                probe_model
                ):
        super().__init__()

        self.ambi_loss_weight = ambi_loss_weight
        self.probe_model = probe_model
        self.probe_model.eval()
        self.llm = AutoModel.from_pretrained(
            llm_pretrained_name,
            output_hidden_states=True
        )

        lora_config = LoraConfig(
            r=lora_r,
            lora_alpha=16,
            lora_dropout=0.05,
            bias="none",
            task_type="FEATURE_EXTRACTION",
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        )

        if use_lora:
            self.llm = get_peft_model(self.llm, lora_config)
        d_model = self.llm.config.hidden_size

        #cats
        cat_cards = [cats_info[c]["cardinality"] for c in cats_info.keys() if cats_info[c]['in_cat']]
        out_cats = [c for c in cats_info if cats_info[c].get('out_cat', False)]
        num_out_cats = len(out_cats)
        cat_emb_dims = [cats_info[c]["emb_dim"] for c in cats_info.keys() if cats_info[c]['in_cat']]  # renamed for clarity

        self.cat_embs = nn.ModuleList([
            nn.Embedding(num_cats, emb_dim)
            for num_cats, emb_dim in zip(cat_cards, cat_emb_dims)
        ])

        late_fusion_width = d_model + sum(cat_emb_dims)
        self.linear_post_fusion = nn.Linear(late_fusion_width, d_model)
        shared_hdim = d_model // 2   # to try: smaller/larger ???
        head_hdim = shared_hdim //2

        layers = []
        in_dim = d_model

        for _ in range(3):
            layers.append(nn.Linear(in_dim, shared_hdim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(p=0.1))
            in_dim = shared_hdim

        self.shared_trunk = nn.Sequential(*layers)
        
        self.bias_head = nn.Sequential(
                nn.Linear(shared_hdim, head_hdim),
                nn.ReLU(),
                nn.Linear(head_hdim, 1)
        )
        
        #after this dude, we add a few more layers on top of this and the shared dim
        
        self.ambi_trunk = nn.Sequential(
            nn.Linear(shared_hdim + 1, shared_hdim),
            nn.ReLU(),
            nn.Linear(shared_hdim, shared_hdim //2),
            nn.ReLU(),
            nn.Linear(shared_hdim //2, shared_hdim //4),
            nn.ReLU()
        )
        
        #same here as the probe
        self.probe_heads = nn.ModuleDict({
            name: nn.Linear(shared_hdim //4, cats_info[name]["cardinality"])
            for name in out_cats
        })

        self.loss_fct_bias = nn.BCEWithLogitsLoss()

    def forward(
        self,
        input_ids,
        attention_mask,
        x_cats,              # [batch, n_cat_vars]
        y_bias=None         # [batch] float {0,1}
    ):

        outputs = self.llm(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        last_hidden = outputs.last_hidden_state # [B, L, d_model]
        mask = attention_mask.unsqueeze(-1)            # [B, L, 1]
        hidden_masked = last_hidden * mask             # [B, L, d_model]
        sum_hidden = hidden_masked.sum(dim=1)          # [B, d_model]
        lengths = mask.sum(dim=1).clamp(min=1)         # [B, 1]
        h_text = sum_hidden / lengths                  # [B, d_model]


        # CATEMBS
        cat_embs = []
        for i, emb_layer in enumerate(self.cat_embs):
            cat_i = x_cats[:, i]           # [B]
            e_i = emb_layer(cat_i)         # [B, emb_dim_i]
            cat_embs.append(e_i)
        e_cat = torch.cat(cat_embs, dim=-1)  # [B, sum(emb_dims)]
        
        h_fused = torch.cat([h_text, e_cat], dim=-1) # [B, late_fusion_width]
        h_proj = self.linear_post_fusion(h_fused)    # [B, d_model]
        h_shared = self.shared_trunk(h_proj)         # [B, shared_hdim]
        
        
        #bias head just sticks on this
        logits_bias = self.bias_head(h_shared).squeeze(-1)   # [B]
        
        #ambi
        h_ambi_base = torch.cat([h_shared, logits_bias.unsqueeze(-1)], dim=-1)
        h_ambi= self.ambi_trunk(h_ambi_base)
        logits_ambi = {name: head(h_ambi) for name, head in self.probe_heads.items()}

        with torch.no_grad():
            logits_ambi_gold = self.probe_model(input_ids, attention_mask, y_bias, out_cats)

        loss_ambi = 0
        for name in self.probe_heads:
            student_log_probs = F.log_softmax(logits_ambi[name], dim=-1)
            teacher_probs = F.softmax(logits_gold[name], dim=-1)
            loss_ambi += F.kl_div(student_log_probs, teacher_probs, reduction='batchmean')
        
        pred_ambi = torch.stack([
            -(F.softmax(logits_ambi[name], dim=-1) * 
              F.log_softmax(logits_ambi[name], dim=-1)).sum(dim=-1) / math.log(cats_info[name]['cardinality'])
            for name in self.probe_heads
        ]).mean(dim=0)  # [B]
        
        loss = None
        loss_bias = self.loss_fct_bias(logits_bias, y_bias)
        loss_ambi = self.ambi_loss_weight * loss_ambi
        loss = loss_bias + loss_ambi

        return {
            "loss": loss,
            "logits_bias": logits_bias,
            "logits_ambi": logits_ambi,
            "pred_ambi" : pred_ambi
        }
        

In [8]:

SUPPORTED_MODELS = [
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "Qwen/Qwen-7B",
    "mediabiasgroup/magpie-pt",
]

REGISTRY_PATH = "./model_registry.json"

def load_registry():
    if not Path(REGISTRY_PATH).exists():
        return []
    with open(REGISTRY_PATH) as f:
        return json.load(f)

def save_to_registry(args, run_name, output_dir, metrics):
    registry = load_registry()
    entry = {
        "run_name": run_name,
        "timestamp": datetime.now().isoformat(),
        "output_dir": output_dir,
        # config
        "model_name": args.model_name,
        "ambi_type": args.ambi_type,
        "epochs": args.epochs,
        "lr": args.lr,
        "use_lora": args.use_lora,
        "lora_r": args.lora_r if args.use_lora else None,
        "zero_cats": args.zero_cats,
        "ambi_loss_weight": args.ambi_loss_weight,
        "seed": args.seed,
        # artifacts
        "weights_path": os.path.join(output_dir, "pytorch_model.bin"),
        "preds_path": os.path.join(output_dir, "test_preds.csv"),
        "metrics_path": os.path.join(output_dir, "metrics.json"),
        "confusion_matrix_path": os.path.join(output_dir, "confusion_matrix.png"),
        # results inline for quick lookup
        "metrics": metrics,
    }
    registry.append(entry)
    with open(REGISTRY_PATH, "w") as f:
        json.dump(registry, f, indent=2)
    print(f"Registered: {run_name}")

def parse_args():
    p = argparse.ArgumentParser(description="Train MTL bias/ambivalence classifier")
    p.add_argument("--model_name",      type=str,   default="meta-llama/Llama-2-7b-hf", choices=SUPPORTED_MODELS)
    p.add_argument("--epochs",          type=int,   default=3)
    p.add_argument("--lr",              type=float, default=5e-5)
    p.add_argument("--batch_size",      type=int,   default=16)
    p.add_argument("--ambi_loss_weight",type=float, default=1.0,  help="Weight on ambivalence loss term")
    p.add_argument("--max_length",      type=int,   default=128)
    p.add_argument("--use_lora",        action="store_true")
    p.add_argument("--lora_r",          type=int,   default=16)
    p.add_argument("--test_size",       type=float, default=0.2)
    p.add_argument("--seed",            type=int,   default=42)
    p.add_argument("--wandb_project",   type=str,   default="bias-ambivalence-mtl")
    p.add_argument("--no_wandb",        action="store_true")
    p.add_argument("--output_dir",      type=str,   default="./model")
    p.add_argument("--test_mode",     type=lambda x: x.lower() == 'true', default=False)
    p.add_argument("--zero_cats", action="store_true", default=False)
    p.add_argument("--probe_model", type=str,   default="probe_training")
    return p.parse_args()


def get_run_name(args):
    model_short = args.model_name.split("/")[-1]
    lora_tag = f"_lora{args.lora_r}" if args.use_lora else "nolora"
    cats_tag = f"_zeroCats" if args.zero_cats else "cats"
    return f"{model_short}_{lora_tag}_{datetime.now().strftime('%m%d_%H%M')}"


def build_compute_metrics():
    acc_metric = evaluate.load("accuracy")
    f1_metric  = evaluate.load("f1")

    def compute_metrics(eval_pred):
        logits_bias, logits_ambi = eval_pred.predictions
        labels_bias, labels_ambi = eval_pred.label_ids

        logits_bias  = np.asarray(logits_bias).reshape(-1)
        logits_ambi  = np.asarray(logits_ambi).reshape(-1)
        labels_bias  = np.asarray(labels_bias).reshape(-1)
        labels_ambi  = np.asarray(labels_ambi).reshape(-1)

        probs_bias = 1 / (1 + np.exp(-logits_bias))
        preds_bias = (probs_bias > 0.5).astype(int)

        mse  = np.mean((logits_ambi - labels_ambi) ** 2)
        rmse = np.sqrt(mse)

        return {
            "bias_acc":  acc_metric.compute(predictions=preds_bias, references=labels_bias)["accuracy"],
            "bias_f1":   f1_metric.compute( predictions=preds_bias, references=labels_bias)["f1"],
            "ambi_rmse": rmse,
        }

    return compute_metrics


def train(args, fdf, cat_embs_info):
    run_name   = get_run_name(args)
    output_dir = os.path.join(args.output_dir, run_name)
    os.makedirs(output_dir, exist_ok=True)

    # --- wandb ---
    report_to = ["none"]
    if not args.no_wandb:
        wandb.init(
            project=args.wandb_project,
            name=run_name,
            config=vars(args),
        )
        report_to = ["wandb"]

    # --- tokenizer & model ---
    tokenizer = AutoTokenizer.from_pretrained(args.model_name, use_fast=False)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        
    probe_model = ambiProbe(cat_embs_info, args.model_name)
    probe_model_state_dir = PROBE_WEIGHTS_DIR + '/' + args.probe_model
    if os.path.exists(probe_model_state_dir):
        state_dict = torch.load(probe_model_state_dir, map_location=device)
        probe_model.load_state_dict(state_dict)
        probe_model.to(device)
    else:
        if not args.probe_model == "probe_training":
            raise Exception('no probe model found for this model type')

    model = MTLClassifier(
        cats_info=cat_embs_info,
        llm_pretrained_name=args.model_name,
        use_lora=args.use_lora,
        lora_r=args.lora_r,
        ambi_loss_weight=args.ambi_loss_weight,
        probe_model=probe_model
    )
    

    # --- data ---
    if args.test_mode:
        ffdf = fdf.sample(1000)
    else:
        ffdf = fdf.copy()
    train_df, test_df = train_test_split(
        ffdf, 
        test_size=args.test_size, 
        random_state=args.seed, 
        shuffle=True
    )
    train_dataset = MTLDataset(train_df, cat_embs_info, args.ambi_type, tokenizer, 
                               max_length=args.max_length, zero_cats=args.zero_cats)
    eval_dataset  = MTLDataset(test_df,  cat_embs_info, args.ambi_type, tokenizer, 
                               max_length=args.max_length, zero_cats=args.zero_cats)

    # --- training args ---
    training_args = TrainingArguments(
        output_dir=output_dir,
        eval_strategy="epoch",
        save_strategy="no",
        learning_rate=args.lr,
        per_device_train_batch_size=args.batch_size,
        per_device_eval_batch_size=args.batch_size * 2,
        num_train_epochs=args.epochs,
        weight_decay=0.01,
        remove_unused_columns=False,
        logging_strategy="steps",
        logging_steps=50,
        save_total_limit=1,
        report_to=report_to,
        label_names=["y_bias", "y_ambi"],
        seed=args.seed,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        tokenizer=tokenizer,
        compute_metrics=build_compute_metrics()
    )

    trainer.train()
    trainer.save_model(output_dir)

    


        # cleanup
    test_preds = trainer.predict(eval_dataset)
    logits_bias, logits_ambi = test_preds.predictions
    probs = 1 / (1 + np.exp(-np.array(logits_bias).reshape(-1)))
    preds = (probs > 0.5).astype(int)
    labels = np.array(test_preds.label_ids[0]).reshape(-1)
    
    pred_df = test_df[["text", "label_binary"]].copy()
    pred_df["pred_bias"] = preds
    pred_df["pred_ambi"] = np.array(logits_ambi).reshape(-1)
    pred_df.to_csv(os.path.join(output_dir, "test_preds.csv"), index=False)
    
    # save confusion matrix
    from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
    import matplotlib.pyplot as plt
    cm = confusion_matrix(labels, preds)
    disp = ConfusionMatrixDisplay(cm, display_labels=["Non-biased", "Biased"])
    disp.plot()
    plt.savefig(os.path.join(output_dir, "confusion_matrix.png"))
    plt.close()
    
    # save metrics
    final_metrics = test_preds.metrics
    with open(os.path.join(output_dir, "metrics.json"), "w") as f:
        json.dump(final_metrics, f, indent=2)
    
    save_to_registry(args, run_name, output_dir, final_metrics)
    if not args.no_wandb:
        wandb.finish()

    return trainer




# Training Probe

In [15]:
import torch
from collections import Counter
from torch.utils.data import DataLoader
from tqdm import tqdm

model.eval()
device = next(model.parameters()).device
out_cat_names = eval_dataset.out_cat_names

marginal_acc = {}
for name in out_cat_names:
    counts = Counter(eval_dataset.out_cat_ids[name].tolist())
    marginal_acc[name] = max(counts.values()) / sum(counts.values())

correct = {name: 0 for name in out_cat_names}
total = 0
loader = DataLoader(eval_dataset, batch_size=64, shuffle=False)

with torch.no_grad():
    for batch in tqdm(loader, desc="Evaluating probe"):
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].to(device)
        x_cats         = batch["x_cats"].to(device)
        target_cats    = batch["target_cats"].to(device)
        
        out = model(input_ids, attention_mask, x_cats, labels, target_cats)
        
        bsz = input_ids.size(0)
        total += bsz
        for cat_idx, name in enumerate(out_cat_names):
            preds = out["out_logits"][name].argmax(dim=-1)
            golds = target_cats[:, cat_idx]
            correct[name] += (preds == golds).sum().item()

print(f"\n{'Variable':<25} {'Probe':>8}  {'Marginal':>10}  {'Lift':>8}")
print("─" * 55)
for name in out_cat_names:
    probe_acc = correct[name] / total
    marg_acc  = marginal_acc[name]
    lift      = probe_acc - marg_acc
    print(f"{name:<25} {probe_acc:>8.3f}  {marg_acc:>10.3f}  {lift:>+8.3f}")

print("─" * 55)
global_probe = sum(correct.values()) / (total * len(out_cat_names))
global_marg  = sum(marginal_acc.values()) / len(out_cat_names)
print(f"{'GLOBAL':<25} {global_probe:>8.3f}  {global_marg:>10.3f}  {global_probe - global_marg:>+8.3f}")

Evaluating probe: 100%|██████████| 622/622 [10:39<00:00,  1.03s/it]


Variable                     Probe    Marginal      Lift
───────────────────────────────────────────────────────
num_foll_news_outlet         0.286       0.306    -0.020
num_poli_idealogy            0.370       0.356    +0.014
num_age                      0.752       0.752    +0.000
num_gender                   0.555       0.563    -0.008
num_education                0.820       0.820    +0.000
───────────────────────────────────────────────────────
GLOBAL                       0.557       0.559    -0.003



═══════════════════════════════════════════════════════════════════════════
  Text:        Before we get to work dispensing these arguments, it’s important to acknowledge that they are, on some level, understandable. A lot of Ameri…
  Bias label:  1
───────────────────────────────────────────────────────────────────────────

  num_foll_news_outlet   gold=0  pred=0  p(pred)=0.29  p(gold)=0.29  ✓
    0  0.29  ██████··············  ← gold/pred
    1  0.21  ████················  
    2  0.25  █████···············  
    3  0.14  ███·················  
    4  0.07  █···················  
    5  0.05  █···················  

  num_poli_idealogy   gold=3  pred=0  p(pred)=0.46  p(gold)=0.11  ✗
    0  0.46  █████████···········  ← pred
    1  0.14  ███·················  
    2  0.18  ████················  
    3  0.11  ██··················  ← gold
    4  0.10  ██··················  
    5  0.00  ····················  

  num_age   gold=1  pred=1  p(pred)=0.62  p(gold)=0.62  ✓
    0  0.00  ·····

In [14]:
#j do this to load



ambiProbe(
  (llm): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (up_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (down_proj): Linear(in_features=5632, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (rotary_emb): LlamaRota

# Training Abalations

- Full model (3 runs, averaged)
- Ambi weight
    - 0.0
    - 0.5
    - 1.0
- magpie-pt
- zero cats
- qwen 

# Inference Abalations
## Does the model work?
- Full model (on hold out set)
- 0.0 ambi weight (on hold out set)
- 0.5 ambi weight (on hold out set)
- 1.0 ambi weight (on hold out set)
- qwen (just hold out)
- magpie-pt (just hold out)
- Full model (zero cats at inference, just hold out)
- zero cats (zero cats at inference, just hold out)

## How does it compare with Spinde
- Full model (3 runs, averaged, full train and test)
- Full model (zero cats at inference, full train and test)
- zero cats (zero cats at inference, full train and test)
- spinde (full train and test)
- magpie-pt (full train and test)

In [ ]:
# # 9. finetuned deberta
sys.argv = [
    "train.py", 
    "--model_name", "mediabiasgroup/magpie-pt", 
    "--test_mode", TEST_MODE, 
    "--lr", "1e-5", 
    "--epochs", "10"]
args = parse_args()
train(args, fdf, cat_embs_info)
import gc

torch.cuda.empty_cache()
gc.collect()

In [ ]:
#1 - full model train
TEST_MODE = "false"
sys.argv = ["train.py", "--model_name", "TinyLlama/TinyLlama-1.1B-Chat-v1.0",  "--test_mode", TEST_MODE]
args = parse_args()
train(args, fdf, cat_embs_info)
import gc
torch.cuda.empty_cache()
gc.collect()